Load the dataset

In [67]:
import pandas as pd
import numpy as np

df = pd.read_csv('Cars Datasets 2025.csv', encoding="latin1")
df.describe(include="all")
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Company Names              1218 non-null   object
 1   Cars Names                 1218 non-null   object
 2   Engines                    1218 non-null   object
 3   CC/Battery Capacity        1215 non-null   object
 4   HorsePower                 1218 non-null   object
 5   Total Speed                1218 non-null   object
 6   Performance(0 - 100 )KM/H  1212 non-null   object
 7   Cars Prices                1218 non-null   object
 8   Fuel Types                 1218 non-null   object
 9   Seats                      1218 non-null   object
 10  Torque                     1217 non-null   object
dtypes: object(11)
memory usage: 104.8+ KB


Initial data quality check

In [68]:
# Missing values
df.isna().sum()

Company Names                0
Cars Names                   0
Engines                      0
CC/Battery Capacity          3
HorsePower                   0
Total Speed                  0
Performance(0 - 100 )KM/H    6
Cars Prices                  0
Fuel Types                   0
Seats                        0
Torque                       1
dtype: int64

In [69]:
# Percent missing values
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

Performance(0 - 100 )KM/H    0.492611
CC/Battery Capacity          0.246305
Torque                       0.082102
Engines                      0.000000
Cars Names                   0.000000
Company Names                0.000000
HorsePower                   0.000000
Total Speed                  0.000000
Cars Prices                  0.000000
Fuel Types                   0.000000
Seats                        0.000000
dtype: float64

In [70]:
# Duplicates
df.duplicated().sum()

np.int64(4)

In [71]:
# Unique values
for col in df.columns:
    print(f'\n{col}')
    print(df[col].unique()[:20])


Company Names
['FERRARI' 'ROLLS ROYCE' 'Ford' 'MERCEDES' 'AUDI' 'BMW' 'ASTON MARTIN'
 'BENTLEY' 'LAMBORGHINI' 'TOYOTA' 'NISSAN' 'ROLLS ROYCE ' 'VOLVO' 'KIA'
 'HONDA' 'KIA  ' 'HYUNDAI' 'MAHINDRA' 'MARUTI SUZUKI' 'Nissan']

Cars Names
['SF90 STRADALE' 'PHANTOM' 'KA+' ' GT 63 S' 'AUDI R8 Gt' 'Mclaren 720s'
 'VANTAGE F1' 'Continental GT Azure' 'VENENO ROADSTER' 'F8 TRIBUTO'
 '812 GTS' 'PORTOFINO' 'ROMA' 'MONZA SP2' 'F8 SPIDER' 'PORTOFINO M'
 'ROMA SPIDER' 'GR SUPRA' 'TOYOTA 86' 'TOYOTA  GR86']

Engines
['V8' 'V12' '1.2L Petrol' 'V10' 'I4' 'BOXER-4' 'V6' 'ELECTRIC MOTOR' 'I6'
 'ELECTRIC ' 'ELECTRIC' 'I3' 'I4 + ELECTRIC' 'HYBRID'
 '1.2L,4-CYLINDER,INLINE-4(I4)' '1.4L,4-CYLINDER,INLINE-4(I4)'
 '2.0L,4-CYLINDER,INLINE-4(I4)' '2.2L,4-CYLINDER,INLINE-4(I4)'
 '1.5L,4-CYLINDER,INLINE(I4)' '2.0L,4-CYLINDER,WITH HYBRID SYSTEM']

CC/Battery Capacity
['3990 cc' '6749 cc' '1,200 cc' '3,982 cc' '5,204 cc' '3,994 cc'
 '3,996 cc' '6,498 cc' '3,900 cc' '6496 cc' '6,496 cc' '2,998 cc'
 '1,998 cc' '2,387 cc

In [72]:
# Convert common representations of missing values to NaN
missing_values = ['-', '--', 'missing', 'null', 'NULL', 'NA', 'N/A', '']
df.replace(missing_values, pd.NA, inplace=True)
df.isna().sum()

Company Names                0
Cars Names                   0
Engines                      0
CC/Battery Capacity          5
HorsePower                   0
Total Speed                  0
Performance(0 - 100 )KM/H    6
Cars Prices                  0
Fuel Types                   0
Seats                        0
Torque                       1
dtype: int64

Data Cleaning

In [73]:
# renaming column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "")
    .str.replace(")", "")
    .str.replace("/", "_")
)
df.columns

Index(['company_names', 'cars_names', 'engines', 'cc_battery_capacity',
       'horsepower', 'total_speed', 'performance0_-_100_km_h', 'cars_prices',
       'fuel_types', 'seats', 'torque'],
      dtype='object')

In [ ]:
# clean numerical values
df['horsepower'].unique()
df["horsepower"] = (
    df["horsepower"]
    .astype(str)
    .str.replace("hp", "", regex=False)
    .str.replace("HP", "", regex=False)
    .str.replace("cc", "", regex=False)
    .str.replace("(est.)", "", regex=False)
    .str.replace("\x96", "-", regex=False)
    .str.replace("Up to", "", regex=False)
    .str.replace("~", "", regex=False)
    .str.replace("/", "-", regex=False)
    .str.strip()
)
def convert_range(value):
    value = str(value).replace(",", "").strip()
    
    if "-" in value:
        parts = value.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return np.nan
    
    try:
        return float(value)
    except:
        return np.nan
df["horsepower"] = df["horsepower"].apply(convert_range)
print(df[df['horsepower'].isnull()])

array(['963 hp', '563 hp', '70-85 hp', '630 hp', '602 hp', '710 hp',
       '656 hp', '550 hp', '750 hp', '789 hp', '592 hp', '612 hp',
       '382 hp', '205 hp', '228 hp', '381 hp', '600 hp', '332 hp',
       '400 hp', '188 hp', '300 hp', '149 hp', '201 hp', '284 hp',
       '310 hp', '1160 hp', '1000 hp', '715 hp', '503 hp', '542 hp',
       '580 hp', '540 hp', '836 hp', '819 hp', '759 hp', '640 hp',
       '740 hp', '641 hp', '610 hp', '700 hp', '769 hp', '671 hp',
       '591 hp', '624 hp', '496 hp', '603 hp', '429 hp', '362 hp',
       '416 hp', '402 hp', '255 hp', '751 hp', '627 hp', '493 hp',
       '444 hp', '523 hp', '623 hp', '335 hp', '349 hp', '306 hp',
       '248 hp', '369 hp', '136 hp', '261 hp', '302 hp', '116 hp',
       '190 hp', '109 hp', '150 hp', '178 hp', '224 hp', '95 hp',
       '102 hp', '163 hp', '247 hp', '295 hp', '187 hp', '240 hp',
       '315 hp', '192 hp', '227 hp', '285 hp', '180 hp', '301 hp',
       '139 hp', '121 hp', '203 hp', '270 hp', '159 hp', '1

In [75]:
#